# Stochastic inventory control by simulation-based policy iteration

A port of `stochinventorycontroSimBasedPI.mlx`.

`A_11_StochInventoryControlPI.ipynb` ran policy iteration with the model in
hand. Here the same loop runs **without one**: the evaluation step is
replaced by simulated trajectories.

The inner loop is a SARSA-style update -- it bootstraps on
$Q(x_{k+1},\mu(x_{k+1}))$, the value of the action the *current policy* would
take, not on a minimum. That is what makes it policy *evaluation* rather
than Q-learning. The outer loop then improves the policy greedily. This is
generalized policy iteration: neither step is exact, and the hope is that the
pair still converges.

Because the exact answer is available from the previous notebook, every
entry can be checked.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (14, 4)

from policy_iteration import sim_based_policy_iteration, policy_iteration

rng = np.random.default_rng(0)

In [ ]:
N, h = 4, 4
n1   = N + 1

def ind2grid(x): return x % n1, x // n1          # -> (stock, stage)
def grid2ind(i, k): return i + k*n1

price, c_item, c_tr, c_hold = 10, 5, 0.5, 0.2
demand  = [2, 2, 1, 1]
salvage = -np.array([0, c_item-c_hold, 2*(c_item-c_hold),
                     3*(c_item-c_hold), 4*(c_item-c_hold)])

def model_and_cost(x, u):
    """The simulator. This is the only access to the model."""
    i, k = ind2grid(x)
    if k == h:
        return x, salvage[i]
    j  = u
    c2 = c_item*j + (c_tr if j > 0 else 0)
    c1 = i*c_hold
    if k != 2:
        return (grid2ind(max(i+j-demand[k], 0), k+1),
                -(min(demand[k], j+i)*price - (c1+c2)))
    if rng.random() > 0.5:                        # stage 3: demand 1 or 0
        return (grid2ind(max(i+j-demand[k], 0), k+1),
                -(min(demand[k], j+i)*price - (c1+c2)))
    return (grid2ind(max(i+j, 0), k+1),
            -(min(0, j+i)*price - (c1+c2)))

num_actions = np.array([1 if ind2grid(x)[1] == h else N+1-ind2grid(x)[0]
                        for x in range(n1*(h+1))])
terminal = list(range(n1*h, n1*(h+1)))

In [ ]:
# the original uses nsim = 300_000 per outer iteration, which takes about a
# minute here; 300_000 is kept so the numbers match the script
Q, mu, J, history = sim_based_policy_iteration(
    num_actions, model_and_cost,
    nsim=300_000, eps=0.8, alpha=0.1,
    prob_init_state=np.r_[np.ones(n1)/n1, np.zeros(n1*h)],
    terminal_states=terminal, outer=10, rng=rng,
    terminal_cost=True, greedy_behaviour=False, eps_decay_at=25_000,
    verbose=False)

J_learned  = np.array([[J[grid2ind(i, k)]  for k in range(h+1)] for i in range(n1)])
mu_learned = np.array([[mu[grid2ind(i, k)] for k in range(h)]   for i in range(n1)])
print("learned J (rows = stock 0..4, columns = stage 0..4):")
print(np.round(J_learned, 2))

In [ ]:
# the exact answer, from the model-based notebook next door
DM = [[[] for _ in range(N+1)] for _ in range(h)]
CM = [[[] for _ in range(N+1)] for _ in range(h)]
PM = [[[] for _ in range(N+1)] for _ in range(h)]
for k in range(h):
    for i in range(N+1):
        if k != 2:
            for j in range(0, N-i+1):
                c2 = c_item*j + (c_tr if j > 0 else 0); c1 = i*c_hold
                DM[k][i].append([max(i+j-demand[k], 0)])
                CM[k][i].append([-(min(demand[k], j+i)*price - (c1+c2))])
                PM[k][i].append([1.0])
        else:
            for j in range(0, max(N+demand[k]-i, N-i) + 1):
                nxt, cst, prb = [], [], []
                for dd_, pr_ in ((demand[k], 0.5), (0, 0.5)):
                    if j <= N + dd_ - i:
                        c2 = c_item*j + (c_tr if j > 0 else 0); c1 = i*c_hold
                        nxt.append(max(i+j-dd_, 0))
                        cst.append(-(min(dd_, j+i)*price - (c1+c2)))
                        prb.append(pr_)
                DM[k][i].append(nxt); CM[k][i].append(cst); PM[k][i].append(prb)

_, J_exact_nested, _ = policy_iteration(
    {"DM": DM, "CM": CM, "PM": PM, "gh": list(salvage)})
J_exact = np.array([[J_exact_nested[k][i] for k in range(h+1)] for i in range(n1)])

print("exact J:");  print(np.round(J_exact, 2))
print("\nerror (learned - exact):"); print(np.round(J_learned - J_exact, 2))
print("\nlearned policy:"); print(mu_learned)

Stage 0 is accurate to a few tenths, and the last stage is exact -- its
Q-factors come straight from the salvage value with no bootstrapping.

The entries still sitting at zero are the interesting ones: high stock at
stages 1 and 2 is **never visited**. Trajectories start at stage 0 and the
policy orders down, so reaching a stock of 3 or 4 mid-horizon would take a
large order into an already-full shop, which the policy never makes. The
exploration rate starts at $0.8$ but drops to $\epsilon/12$ after the first
$25\,000$ simulations, so chance stops doing it either.

This is the same failure the Q-learning notebook in `code/unit10/` shows,
and it is the first condition of the convergence theorem -- every
state-action pair infinitely often -- visibly not holding. Here it matters
less than it looks: those states are unreachable under any good policy, so
not learning them costs nothing. That is the point the unit's "Exploration"
slide makes with this very example.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(n1):
    ax.plot([J_l[grid2ind(i, 0)] for _, J_l in history], "o-", label=f"stock {i}")
    ax.axhline(J_exact[i, 0], color="k", ls=":", lw=0.8)
ax.set_xlabel("outer (policy improvement) iteration")
ax.set_ylabel("$J$ at stage 0"); ax.grid(True); ax.legend(fontsize=8)
ax.set_title("dotted lines: the exact values")
plt.tight_layout(); plt.show()